In [10]:
%reload_ext autoreload
%autoreload 2

import pynq
import numpy as np
import time

# Reset PL
pynq.PL.reset()

# Load bitstream
ol = pynq.Overlay('pll_final_version.bit')

In [15]:
fs = 125e6
FCW = 46

def set_frequency(freq):  # free running frequency. Used when pll is not enabled
    ftw = int(freq * (2**FCW) / fs)

    ftw_low  = ftw & 0xFFFFFFFF
    ftw_high = (ftw >> 32) & 0x3FFF

    ol.reg_bank_0.write(0x60, ftw_low)
    ol.reg_bank_0.write(0x64, ftw_high)

    print(f"FTW: {ftw}")

def set_amplitude(norm_amp):
    MAX_AMP = (2**15 - 1)
    val = int(norm_amp * MAX_AMP)
    # clamp just in case
    val = max(0, min(val, MAX_AMP))
    ol.reg_bank_0.write(0x40, val)
    print(f"Amplitude set to {val} (normalized {norm_amp})")

def use_pll(enable):
    ol.reg_bank_0.write(0x44, int(enable))
    
def set_lpf_coefficients(cutoff_freq):
    """
    Calculates and uploads the fixed-point low-pass filter coefficients
    """
    # 1. Calculate continuous-time alpha floating point parameter
    alpha = 1.0 - np.exp(-2.0 * np.pi * cutoff_freq / fs)
    
    # 2. Scale alpha up into fixed-point representations
    # lpf_kk_i uses Q1.24 precision format
    kk_fixed = int(alpha * (2**24))
    
    # lpf_aa_i uses Q1.25 precision format
    aa_fixed = int(alpha * (2**25))
    
    # 3. Apply bounds check for hardware registers
    # lpf_aa_i is constrained to a signed 18-bit input on the hardware module.
    # We drop the 7 lower bits of the 25-bit scale to tightly map into an 18-bit bounds check.
    aa_reg_val = max(0, min(aa_fixed, (2**17) - 1))
    kk_reg_val = max(0, min(kk_fixed, (2**24) - 1))
    pp_reg_val = 0 # Pre-configured to 0 per testbench specifications
    
    # 4. Write coefficients out via AXI-lite register bank
    ol.reg_bank_0.write(0x54, aa_reg_val) # lpf_aa_i
    ol.reg_bank_0.write(0x58, pp_reg_val) # lpf_pp_i
    ol.reg_bank_0.write(0x5C, kk_reg_val) # lpf_kk_i
    
    print(f"LPF Configured for Cutoff: {cutoff_freq} Hz")
    print(f" -> AA Reg Val (18-bit scaled): {aa_reg_val}")
    print(f" -> KK Reg Val (Q24):            {kk_reg_val}")

In [16]:
set_frequency(67859)
use_pll(1)
set_amplitude(1)

# Configure the LPF cutoff filter parameter (e.g., 1000 Hz)
set_lpf_coefficients(1000.0)

FTW: 38201220889
Amplitude set to 32767 (normalized 1)
LPF Configured for Cutoff: 1000.0 Hz
 -> AA Reg Val (18-bit scaled): 1686
 -> KK Reg Val (Q24):            843


In [17]:
dma_main = ol.axi_dma_0
dma_freq = ol.axi_dma_1

N = 1000000  # number of samples

# Allocate buffer (64-bit words)
buf_main = pynq.allocate(shape=(N,), dtype=np.uint64)
buf_freq = pynq.allocate(shape=(N,), dtype=np.uint64)

ol.reg_bank_0.write(0x68, N)

# Start DMA transfer
dma_main.recvchannel.transfer(buf_main)
dma_freq.recvchannel.transfer(buf_freq)

ol.reg_bank_0.write(0x6C, 1)
ol.reg_bank_0.write(0x6C, 0)

dma_main.recvchannel.wait()
dma_freq.recvchannel.wait()

print("DMA capture complete")

DMA capture complete


In [18]:
from datetime import datetime

data_main = np.array(buf_main)
data_freq = np.array(buf_freq)
# fname = f"pll_dump_coeff_change_{datetime.now().strftime('%Y%m%d_%H%M%S')}.npz"
fname = f"pll_final_version_{datetime.now().strftime('%Y%m%d_%H%M%S')}.npz"


# Save data + metadata
np.savez(
    fname,
    data_main=data_main,
    data_freq=data_freq,
    fs=fs,
    N=N
)

print(f"Saved {fname}")

Saved pll_final_version_20221009_080347.npz
